# Test Inference and Evaluation

This notebook evaluates the fine-tuned XCrime-LLM model on the full NYC test split. It constructs the multi-label prediction prompts, performs model inference, and reports the prediction and latency metrics used for evaluation.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# Install the OpenAI package version used by the inference code
!pip install -q openai==0.28

In [ ]:
import os
import openai

# Set your OpenAI API key as an environment variable before running this notebook
openai.api_key = os.environ["OPENAI_API_KEY"]

### Configure Evaluation Pipeline

Import the utilities required for asynchronous inference, output validation, and evaluation of the hard 0/1 ANY-in-7 predictions.

In [ ]:
import os
import json
import time
import math
import asyncio
import random

from asyncio import as_completed
from typing import Iterable, Optional, Tuple, Dict, Any

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    accuracy_score,
)

from jsonschema import Draft202012Validator

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.6f}",
)

### Configure Model Inference

Define the test-data location, fine-tuned model, crime categories, prediction horizon, and inference settings used for the XCrime-LLM evaluation.

In [ ]:
# Data and model configuration
DATA_DIR = "/content/drive/MyDrive/XCrime-LLM/data/splits"
RESULTS_DIR = "/content/drive/MyDrive/XCrime-LLM/results"

os.makedirs(RESULTS_DIR, exist_ok=True)

# Set your fine-tuned model ID as an environment variable before running
MODEL_NAME = os.environ["XCRIME_MODEL_NAME"]

CRIMES = [
    "BURGLARY",
    "ROBBERY",
    "GRAND LARCENY",
    "FELONY ASSAULT",
]

HIST = 0
FUT = 7
RANDOM = 42

# Concurrency and request settings
MAX_CONCURRENCY = 10
MAX_RETRIES = 0
REQUEST_PAUSE = 0.02
TIMEOUT_SEC = 45
WAVE_SIZE = 2000
PROGRESS_DIV = 200

# Deterministic single-pass generation
TEMP = 0
TOP_P = 1.0
MAX_TOK = 40

np.random.seed(RANDOM)
random.seed(RANDOM)

### Apply Input Guardrails

Standardize the test-set fields, constrain feature values to the ranges used by XCrime-LLM, enforce valid region-date keys, and retain only the features required for inference.

In [ ]:
def apply_input_guardrails(
    df: pd.DataFrame,
    crimes: Iterable[str],
    hist: int,
    fut: int,
    date_col: str = "date",
    region_col: str = "region_id",
    clip_from: Optional[pd.DataFrame] = None,
    date_bounds: Optional[Tuple[str, str]] = None,
    quantile_hi: float = 0.995,
    drop_if_missing_required: bool = False,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:

    CAP_RECENCY = 365
    RECENCY_SENTINEL = 9999

    report: Dict[str, Any] = {
        "missing_required_cols": [],
        "rows_dropped_missing_key": 0,
        "rows_dropped_bad_date": 0,
        "rows_kept": 0,
        "notes": [],
    }

    x = df.copy()

    # Required input columns
    required = [
        date_col,
        region_col,
        "crime",
        "last7_total",
        "last28_mean",
        "recency",
        "R1_influence",
        "base_rate",
        "dow",
        "month",
    ]

    missing = [
        column
        for column in required
        if column not in x.columns
    ]

    if missing and drop_if_missing_required:
        raise KeyError(
            f"Missing required columns: {missing}"
        )

    if missing:
        report["missing_required_cols"] = missing

    # Standardize key fields
    x[date_col] = pd.to_datetime(
        x.get(date_col),
        errors="coerce",
    )

    x[region_col] = (
        pd.to_numeric(
            x.get(region_col),
            errors="coerce",
        )
        .astype("Int64")
    )

    x["crime"] = (
        x.get("crime", "")
        .astype("string")
        .str.strip()
    )

    key_ok = (
        x[date_col].notna()
        & x[region_col].notna()
        & x["crime"].notna()
    )

    report["rows_dropped_missing_key"] = int(
        (~key_ok).sum()
    )

    x = x.loc[key_ok].copy()

    # Standardize numerical features
    for column in [
        "last7_total",
        "last28_mean",
        "R1_influence",
        "base_rate",
    ]:
        if column in x.columns:
            x[column] = (
                pd.to_numeric(
                    x[column],
                    errors="coerce",
                )
                .fillna(0.0)
                .astype(float)
            )

    if "base_rate" in x.columns:
        x["base_rate"] = x["base_rate"].clip(
            0.0,
            1.0,
        )

    # Apply recency sentinel and cap
    if "recency" in x.columns:
        x["recency"] = pd.to_numeric(
            x["recency"],
            errors="coerce",
        )

        x["recency"] = np.where(
            x["recency"].isna(),
            RECENCY_SENTINEL,
            x["recency"],
        )

        x["recency"] = np.where(
            x["recency"] == RECENCY_SENTINEL,
            RECENCY_SENTINEL,
            np.minimum(
                x["recency"],
                CAP_RECENCY,
            ),
        ).astype(int)

    # Standardize calendar features
    for column in ["dow", "month"]:
        if column in x.columns:
            x[column] = (
                pd.to_numeric(
                    x[column],
                    errors="coerce",
                )
                .fillna(0)
                .astype(int)
            )

    # Ensure the prediction horizon fits within the date window
    if date_bounds is not None:
        start_dt, end_dt = map(
            pd.to_datetime,
            date_bounds,
        )

        window_end = (
            x[date_col]
            + pd.to_timedelta(
                fut - 1,
                unit="D",
            )
        )

        valid_date = (
            (x[date_col] >= start_dt)
            & (window_end <= end_dt)
        )

        report["rows_dropped_bad_date"] = int(
            (~valid_date).sum()
        )

        x = x.loc[valid_date].copy()

    # Retain only inference features and the optional evaluation label
    keep_cols = [
        region_col,
        date_col,
        "crime",
        "last7_total",
        "last28_mean",
        "recency",
        "R1_influence",
        "base_rate",
        "dow",
        "month",
    ]

    if "label_7d" in x.columns:
        keep_cols.append("label_7d")

    x = x[
        [
            column
            for column in keep_cols
            if column in x.columns
        ]
    ].copy()

    report["rows_kept"] = len(x)

    return x, report


def pre_prompt_row_asserts(
    row: pd.Series,
    crimes: Iterable[str],
    hist: int,
    date_col: str = "date",
    region_col: str = "region_id",
) -> None:

    if (
        pd.isna(row[date_col])
        or pd.isna(row[region_col])
        or pd.isna(row.get("crime"))
    ):
        raise ValueError(
            "Missing key fields after guardrails."
        )

### Load and Prepare the Test Set

Load the full NYC test split, identify and standardize the date field, apply the input guardrails, and retain the valid region-date anchors used for evaluation.

In [ ]:
TEST_PATH = f"{DATA_DIR}/master_test.csv"

test = pd.read_csv(TEST_PATH)

# Detect the date column
if "complaint_dt" in test.columns:
    DATE_COL = "complaint_dt"
elif "date" in test.columns:
    DATE_COL = "date"
else:
    raise KeyError(
        "Expected a date column named 'complaint_dt' or 'date' in the test file."
    )

# Standardize dates
test[DATE_COL] = pd.to_datetime(
    test[DATE_COL],
    errors="coerce",
)

date_min = test[DATE_COL].min()
date_max = test[DATE_COL].max()

if pd.isna(date_min) or pd.isna(date_max):
    raise ValueError(
        "No valid dates were found in the test file."
    )

# Apply the same input guardrails used by the evaluation pipeline
test_clean, guard_report = apply_input_guardrails(
    df=test,
    crimes=CRIMES,
    hist=HIST,
    fut=FUT,
    date_col=DATE_COL,
    region_col="region_id",
    clip_from=None,
    date_bounds=(
        date_min.date().isoformat(),
        date_max.date().isoformat(),
    ),
    quantile_hi=0.995,
    drop_if_missing_required=False,
)

sel_df = test_clean.reset_index(drop=True)

# Count valid region-date anchors
anchors_valid = (
    sel_df
    .groupby(["region_id", DATE_COL])
    .ngroups
)

print(
    f"[TEST] rows_in={len(test):,} "
    f"| rows_valid={len(sel_df):,} "
    f"| anchors_valid={anchors_valid:,}",
    flush=True,
)

print(
    "[TEST] guardrails report:",
    guard_report,
    flush=True,
)

### Align Test Data for Inference

Map the four crime categories to their canonical event types, standardize the inference features, and verify that each region-date anchor contains all four event rows.

In [ ]:
# Ensure a canonical 'date' column exists
if DATE_COL != "date":
    sel_df["date"] = pd.to_datetime(
        sel_df[DATE_COL],
        errors="coerce",
    )
else:
    sel_df["date"] = pd.to_datetime(
        sel_df["date"],
        errors="coerce",
    )


# Canonical event categories
EVENT_CATEGORIES = [
    "EVENT_TYPE_A",
    "EVENT_TYPE_B",
    "EVENT_TYPE_C",
    "EVENT_TYPE_D",
]

event_mapping = {
    "BURGLARY": "EVENT_TYPE_A",
    "ROBBERY": "EVENT_TYPE_B",
    "GRAND LARCENY": "EVENT_TYPE_C",
    "FELONY ASSAULT": "EVENT_TYPE_D",
}

reverse_mapping = {
    event_type: crime
    for crime, event_type in event_mapping.items()
}


# Create or normalize the event-type column
if "event_type" not in sel_df.columns:
    if "event_column" in sel_df.columns:
        sel_df = sel_df.rename(
            columns={
                "event_column": "event_type"
            }
        )
    else:
        sel_df["event_type"] = (
            sel_df["crime"]
            .astype("string")
            .str.upper()
            .str.strip()
            .map(event_mapping)
        )

sel_df["event_type"] = (
    sel_df["event_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

unique_events = set(
    sel_df["event_type"]
    .dropna()
    .unique()
)

if not unique_events.issubset(
    set(EVENT_CATEGORIES)
):
    sel_df["event_type"] = (
        sel_df["event_type"]
        .map(event_mapping)
    )


# Reconstruct crime names if needed
if "crime" not in sel_df.columns:
    current_events = set(
        sel_df["event_type"]
        .dropna()
        .unique()
    )

    if current_events.issubset(
        set(EVENT_CATEGORIES)
    ):
        sel_df["crime"] = (
            sel_df["event_type"]
            .map(reverse_mapping)
        )
    else:
        raise ValueError(
            "Missing 'crime' and cannot reconstruct it from event_type."
        )


# Normalize neighboring-region feature name if needed
if (
    "R1_pressure" in sel_df.columns
    and "R1_influence" not in sel_df.columns
):
    sel_df = sel_df.rename(
        columns={
            "R1_pressure": "R1_influence"
        }
    )


# Standardize numerical inference features
for column in [
    "last7_total",
    "last28_mean",
    "R1_influence",
    "base_rate",
]:
    if column in sel_df.columns:
        sel_df[column] = (
            pd.to_numeric(
                sel_df[column],
                errors="coerce",
            )
            .fillna(0.0)
            .astype(float)
        )


# Preserve the recency sentinel and cap non-sentinel values
if "recency" in sel_df.columns:
    sel_df["recency"] = pd.to_numeric(
        sel_df["recency"],
        errors="coerce",
    )

    sel_df["recency"] = np.where(
        sel_df["recency"].isna(),
        9999,
        sel_df["recency"],
    )

    sel_df["recency"] = np.where(
        sel_df["recency"] == 9999,
        9999,
        np.minimum(
            sel_df["recency"],
            365,
        ),
    ).astype(int)


# Keep base rate within [0, 1]
if "base_rate" in sel_df.columns:
    sel_df["base_rate"] = (
        pd.to_numeric(
            sel_df["base_rate"],
            errors="coerce",
        )
        .fillna(0.0)
        .clip(0.0, 1.0)
    )


# Standardize calendar features
for column in ["dow", "month"]:
    if column in sel_df.columns:
        sel_df[column] = (
            pd.to_numeric(
                sel_df[column],
                errors="coerce",
            )
            .fillna(0)
            .astype(int)
        )


# Retain only the fields required downstream
KEEP = [
    "region_id",
    "date",
    "crime",
    "event_type",
    "last7_total",
    "last28_mean",
    "recency",
    "R1_influence",
    "base_rate",
    "dow",
    "month",
]

extra_date_col = (
    [DATE_COL]
    if DATE_COL != "date"
    and DATE_COL in sel_df.columns
    else []
)

optional_cols = (
    ["label_7d"]
    if "label_7d" in sel_df.columns
    else []
)

kept_cols = (
    [
        column
        for column in KEEP
        if column in sel_df.columns
    ]
    + extra_date_col
    + optional_cols
)

sel_df = sel_df[
    kept_cols
].copy()


# Each region-date anchor must contain all four event types
event_counts = (
    sel_df
    .groupby(
        ["region_id", "date"]
    )["event_type"]
    .nunique()
)

assert (
    event_counts == 4
).all(), (
    "Some anchors are missing one or more event types."
)

print(
    f"[OK] columns={list(sel_df.columns)} "
    f"| rows={len(sel_df):,}"
)

### Construct Test-Time Prompts

Define the system message and compact multi-label prompt format used for inference, matching the prompt structure used to create the fine-tuning data.

In [ ]:
assert "sel_df" in globals(), (
    "Expected the cleaned test DataFrame 'sel_df'."
)

# Fixed output order
EVENT_ORDER = EVENT_CATEGORIES.copy()

# Feature configuration
USE_BASE_RATE = True
USE_R1_INFL = True
USE_M28 = True
SHOW_ANCHOR_META = True


def _make_schema_json(events):
    return (
        "{"
        + ",".join(
            f'"{event_type}":0'
            for event_type in events
        )
        + "}"
    )


def _build_system_msg(events):
    event_list = ", ".join(events)
    schema_json = _make_schema_json(events)

    content = (
        "You are a spatiotemporal analyst.\n"
        f"For the SAME (REGION, DATE), output independent 0/1 forecasts for each of: {event_list} "
        "for whether ≥1 incident will occur in the NEXT 7 DAYS.\n"
        "Rules: decide independently; use ONLY provided numeric features; no outside knowledge; "
        "output JSON ONLY (no prose).\n"
        "Feature notes: last7_total↑, last28_mean↑, base_rate↑, R1_influence↑ ⇒ higher risk; "
        "recency lower ⇒ higher risk (9999=never). dow=0–6, month=1–12.\n"
        "Return ONE compact JSON object with exactly these keys and 0/1 values:\n"
        f"{schema_json}\n"
    )

    return {
        "role": "system",
        "content": content,
    }


SYSTEM_MSG = _build_system_msg(
    EVENT_ORDER
)

SCHEMA_JSON = _make_schema_json(
    EVENT_ORDER
)


def _g(value, default):
    try:
        if pd.isna(value):
            return default
        return value
    except Exception:
        return default


def _fmt_i(value) -> str:
    try:
        return str(int(value))
    except Exception:
        return "0"


def _fmt_f(value, nd=4) -> str:
    try:
        value = float(value)

        if np.isfinite(value):
            return f"{value:.{nd}f}"

        return f"{0.0:.{nd}f}"

    except Exception:
        return f"{0.0:.{nd}f}"


def _derive_feats_raw(
    row: pd.Series,
) -> dict:

    features = {
        "last7_total": int(
            max(
                0,
                int(
                    _g(
                        row.get(
                            "last7_total",
                            0,
                        ),
                        0,
                    )
                ),
            )
        ),
        "recency": int(
            max(
                0,
                int(
                    _g(
                        row.get(
                            "recency",
                            9999,
                        ),
                        9999,
                    )
                ),
            )
        ),
    }

    if USE_BASE_RATE:
        features["base_rate"] = float(
            min(
                max(
                    float(
                        _g(
                            row.get(
                                "base_rate",
                                0.0,
                            ),
                            0.0,
                        )
                    ),
                    0.0,
                ),
                1.0,
            )
        )

    if USE_R1_INFL:
        features["R1_influence"] = float(
            max(
                0.0,
                float(
                    _g(
                        row.get(
                            "R1_influence",
                            0.0,
                        ),
                        0.0,
                    )
                ),
            )
        )

    if USE_M28:
        features["last28_mean"] = float(
            max(
                0.0,
                float(
                    _g(
                        row.get(
                            "last28_mean",
                            0.0,
                        ),
                        0.0,
                    )
                ),
            )
        )

    return features


def _event_line(
    event_type: str,
    features: dict,
) -> str:

    parts = [
        (
            "last7_total="
            f"{_fmt_i(features.get('last7_total', 0))}"
        ),
        (
            "recency="
            f"{_fmt_i(features.get('recency', 9999))}"
        ),
    ]

    if USE_BASE_RATE:
        parts.append(
            "base_rate="
            f"{_fmt_f(features.get('base_rate', 0.0), 4)}"
        )

    if USE_R1_INFL:
        parts.append(
            "R1_influence="
            f"{_fmt_f(features.get('R1_influence', 0.0), 3)}"
        )

    if USE_M28:
        parts.append(
            "last28_mean="
            f"{_fmt_f(features.get('last28_mean', 0.0), 3)}"
        )

    return (
        f"{event_type}: "
        + " ".join(parts)
    )


def build_multi_prompt(
    group_df: pd.DataFrame,
) -> str:

    region_id = int(
        group_df["region_id"].iloc[0]
    )

    anchor_date = pd.to_datetime(
        group_df[DATE_COL].iloc[0],
        errors="coerce",
    )

    date_string = (
        str(anchor_date.date())
        if pd.notna(anchor_date)
        else str(
            group_df[DATE_COL].iloc[0]
        )
    )

    dow = (
        int(anchor_date.weekday())
        if pd.notna(anchor_date)
        else int(
            group_df["dow"].iloc[0]
            if "dow" in group_df
            else -1
        )
    )

    month = (
        int(anchor_date.month)
        if pd.notna(anchor_date)
        else int(
            group_df["month"].iloc[0]
            if "month" in group_df
            else 0
        )
    )

    # Select one row for each event type
    rows_by_event = {}

    for event_type in EVENT_ORDER:
        mask = (
            group_df["event_type"]
            == event_type
        )

        rows_by_event[event_type] = (
            group_df.loc[mask]
            .head(1)
            .iloc[0]
            if mask.any()
            else None
        )

    features_by_event = {
        event_type: _derive_feats_raw(row)
        for event_type, row
        in rows_by_event.items()
        if row is not None
    }

    lines = []

    if SHOW_ANCHOR_META:
        lines.append(
            f"RID={region_id};"
            f"DT={date_string};"
            f"D={dow};"
            f"M={month}"
        )

    for event_type in EVENT_ORDER:
        features = features_by_event.get(
            event_type
        )

        if features is None:
            base = (
                "last7_total=0 "
                "recency=9999"
            )

            if USE_BASE_RATE:
                base += (
                    " base_rate=0.0000"
                )

            if USE_R1_INFL:
                base += (
                    " R1_influence=0.000"
                )

            if USE_M28:
                base += (
                    " last28_mean=0.000"
                )

            lines.append(
                f"{event_type}: {base}"
            )

        else:
            lines.append(
                _event_line(
                    event_type,
                    features,
                )
            )

    return "\n".join(lines)


# Build one prompt for each region-date anchor
key_cols = [
    "region_id",
    DATE_COL,
]

grouped = sel_df.groupby(
    key_cols,
    sort=False,
)

multi_rows = []

for (region_id, anchor_date), group in grouped:
    prompt = build_multi_prompt(
        group
    )

    multi_rows.append(
        {
            "region_id": int(
                region_id
            ),
            DATE_COL: pd.to_datetime(
                anchor_date,
                errors="coerce",
            ),
            "multi_prompt": prompt,
        }
    )

anchors_ml = pd.DataFrame(
    multi_rows
)

print(
    f"Total multi-label anchors: "
    f"{len(anchors_ml):,}"
)

print(
    "\nSample compact prompt:\n",
    anchors_ml["multi_prompt"].iloc[0],
)

### Parse and Validate Model Outputs

Define the expected four-label output schema and robustly parse model responses into binary predictions for the four event categories.

In [ ]:
import re

# Concurrency semaphore used by the inference pipeline
sem = asyncio.Semaphore(MAX_CONCURRENCY)

# Expected output keys in fixed order
EXPECTED_KEYS = [
    "EVENT_TYPE_A",
    "EVENT_TYPE_B",
    "EVENT_TYPE_C",
    "EVENT_TYPE_D",
]


# JSON schema for the expected prediction object
_SCHEMA = {
    "type": "object",
    "properties": {
        key: {
            "type": "integer",
            "enum": [0, 1],
        }
        for key in EXPECTED_KEYS
    },
    "required": EXPECTED_KEYS,
    "additionalProperties": False,
}

_validator = Draft202012Validator(
    _SCHEMA
)


# Extract a JSON object embedded in surrounding text
_JSONOBJ_RE = re.compile(
    r"\{.*\}",
    re.S,
)


def _strip_code_fence(text: str) -> str:
    """
    Remove Markdown code fences and an optional language tag.
    """
    text = (text or "").strip()

    if text.startswith("```"):
        text = text.strip("`")
        lines = text.splitlines()

        if (
            lines
            and lines[0]
            and not lines[0].strip().startswith("{")
            and not lines[0].strip().startswith("[")
        ):
            text = "\n".join(
                lines[1:]
            )
        else:
            text = "\n".join(
                lines
            )

    return text.strip()


def _coerce_bit(value) -> int:
    try:
        return (
            1
            if int(value) == 1
            else 0
        )
    except Exception:
        return 0


def _ensure_4_keys(data: dict) -> dict:
    """
    Ensure all four expected keys exist and contain binary values.
    Missing keys are filled with 0 and extra keys are ignored.
    """
    return {
        key: _coerce_bit(
            data.get(key, 0)
        )
        for key in EXPECTED_KEYS
    }


def parse_multi_json(text: str) -> dict:
    """
    Parse model output into exactly four binary event predictions.

    Accepted forms include:
    - Plain JSON object
    - Code-fenced JSON
    - {"predictions": {...}} wrapper
    - Four-item list in EVENT_TYPE_A-D order

    Parsing failures fall back to all-zero predictions.
    """
    zeros = {
        key: 0
        for key in EXPECTED_KEYS
    }

    if not text:
        return zeros

    cleaned = _strip_code_fence(
        text
    )

    match = _JSONOBJ_RE.search(
        cleaned
    )

    candidate = (
        match.group(0)
        if match
        else cleaned
    )

    # Try JSON object first
    try:
        obj = json.loads(
            candidate
        )

    except Exception:
        # Fall back to a four-item list
        try:
            list_candidate = (
                candidate.strip()
            )

            if not list_candidate.startswith("["):
                list_candidate = (
                    "["
                    + list_candidate
                    .strip()
                    .strip(",")
                    + "]"
                )

            values = json.loads(
                list_candidate
            )

            if (
                isinstance(values, list)
                and len(values) >= 4
            ):
                return {
                    key: _coerce_bit(value)
                    for key, value
                    in zip(
                        EXPECTED_KEYS,
                        values[:4],
                    )
                }

            return zeros

        except Exception:
            return zeros

    # Unwrap {"predictions": {...}}
    if (
        isinstance(obj, dict)
        and "predictions" in obj
        and isinstance(
            obj["predictions"],
            dict,
        )
    ):
        obj = obj[
            "predictions"
        ]

    # Dictionary output
    if isinstance(obj, dict):
        predictions = (
            _ensure_4_keys(obj)
        )

        # Soft schema validation
        try:
            _validator.validate(
                predictions
            )
        except Exception:
            predictions = (
                _ensure_4_keys(
                    predictions
                )
            )

        return predictions

    # List output
    if (
        isinstance(obj, list)
        and len(obj) >= 4
    ):
        return {
            key: _coerce_bit(value)
            for key, value
            in zip(
                EXPECTED_KEYS,
                obj[:4],
            )
        }

    return zeros


def safe_parse_with_guard(
    raw: str,
) -> dict:
    """
    Parse the response and guarantee four binary predictions.
    """
    predictions = parse_multi_json(
        raw
    )

    for key in EXPECTED_KEYS:
        predictions[key] = (
            1
            if predictions.get(
                key,
                0,
            ) == 1
            else 0
        )

    return predictions


def pretty_json(
    data: dict,
) -> str:
    """Return compact JSON for debugging."""
    try:
        return json.dumps(
            data,
            ensure_ascii=False,
            separators=(",", ":"),
        )
    except Exception:
        return str(data)

### Run Concurrent Model Inference

Define the asynchronous inference pipeline used to generate one hard 0/1 multi-label prediction per test anchor, including format enforcement, fallback requests, concurrency control, and latency recording.

In [ ]:
# Output-format reminders appended to the user prompt
TAIL_PRIMARY = (
    "Return ONLY a compact JSON object with EXACTLY these four keys "
    "and 0/1 integers. No prose."
)

TAIL_FALLBACK = (
    'Output EXACTLY this minified JSON with 0/1 integers and NO prose: '
    '{"EVENT_TYPE_A":0,"EVENT_TYPE_B":0,'
    '"EVENT_TYPE_C":0,"EVENT_TYPE_D":0}'
)


async def _call_once_multi(prompt: str):
    """
    Run one model-inference request.

    Returns:
        (
            {"preds": {...}, "raw": str, "err": bool},
            ttft_ms,
            e2e_ms,
            request_id,
        )
    """

    start_time = time.perf_counter()
    request_id = None
    content = ""

    predictions = {
        "EVENT_TYPE_A": 0,
        "EVENT_TYPE_B": 0,
        "EVENT_TYPE_C": 0,
        "EVENT_TYPE_D": 0,
    }

    async def _make_call(
        system_message: str,
        user_message: str,
        max_tokens: int,
    ):
        return await openai.ChatCompletion.acreate(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": system_message,
                },
                {
                    "role": "user",
                    "content": user_message,
                },
            ],
            temperature=TEMP,
            top_p=TOP_P,
            max_tokens=max_tokens,
            timeout=TIMEOUT_SEC,
            stream=False,
        )

    try:
        # Primary request
        response = await _make_call(
            SYSTEM_MSG["content"],
            prompt + "\n" + TAIL_PRIMARY,
            max_tokens=MAX_TOK,
        )

        first_response_time = time.perf_counter()

        message = response["choices"][0]["message"]
        request_id = response.get("id")

        content = (
            message.get("content") or ""
        ).strip()

        predictions = parse_multi_json(
            content
        )

        # Preserve the original completeness check
        present_keys = sum(
            1
            for key in EXPECTED_KEYS
            if key in content
        )

        need_retry = (
            present_keys < 4
            or sum(
                int(value)
                for value in predictions.values()
            ) == 0
        )

        if need_retry:
            retry_response = await _make_call(
                SYSTEM_MSG["content"],
                prompt + "\n" + TAIL_FALLBACK,
                max_tokens=max(
                    48,
                    MAX_TOK,
                ),
            )

            retry_message = (
                retry_response["choices"][0]["message"]
            )

            request_id = (
                request_id
                or retry_response.get("id")
            )

            retry_content = (
                retry_message.get("content") or ""
            ).strip()

            retry_predictions = parse_multi_json(
                retry_content
            )

            retry_present_keys = sum(
                1
                for key in EXPECTED_KEYS
                if key in retry_content
            )

            if (
                retry_present_keys > present_keys
                or sum(
                    int(value)
                    for value in retry_predictions.values()
                )
                > sum(
                    int(value)
                    for value in predictions.values()
                )
            ):
                content = retry_content
                predictions = retry_predictions

    except Exception:
        # Single fallback request
        try:
            response = await _make_call(
                SYSTEM_MSG["content"],
                prompt + "\n" + TAIL_FALLBACK,
                max_tokens=max(
                    48,
                    MAX_TOK,
                ),
            )

            first_response_time = (
                time.perf_counter()
            )

            message = (
                response["choices"][0]["message"]
            )

            request_id = response.get("id")

            content = (
                message.get("content") or ""
            ).strip()

            predictions = parse_multi_json(
                content
            )

        except Exception:
            end_time = time.perf_counter()

            ttft_ms = max(
                0.1,
                (
                    end_time
                    - start_time
                )
                * 1000.0,
            )

            e2e_ms = ttft_ms

            return (
                {
                    "preds": predictions,
                    "raw": "",
                    "err": True,
                },
                ttft_ms,
                e2e_ms,
                request_id,
            )

    end_time = time.perf_counter()

    ttft_ms = max(
        0.1,
        (
            first_response_time
            - start_time
        )
        * 1000.0,
    )

    e2e_ms = max(
        0.1,
        (
            end_time
            - start_time
        )
        * 1000.0,
    )

    # Enforce hard binary predictions
    predictions = {
        key: (
            1
            if int(value) == 1
            else 0
        )
        for key, value
        in predictions.items()
    }

    return (
        {
            "preds": predictions,
            "raw": content,
            "err": False,
        },
        ttft_ms,
        e2e_ms,
        request_id,
    )


async def _call_with_retry_multi(
    prompt: str,
):
    for attempt in range(
        MAX_RETRIES + 1
    ):
        try:
            async with sem:
                if REQUEST_PAUSE:
                    await asyncio.sleep(
                        REQUEST_PAUSE
                        + random.uniform(
                            0,
                            0.01,
                        )
                    )

                return await asyncio.wait_for(
                    _call_once_multi(
                        prompt
                    ),
                    timeout=TIMEOUT_SEC + 5,
                )

        except Exception:
            if attempt == MAX_RETRIES:
                return (
                    {
                        "preds": {
                            "EVENT_TYPE_A": 0,
                            "EVENT_TYPE_B": 0,
                            "EVENT_TYPE_C": 0,
                            "EVENT_TYPE_D": 0,
                        },
                        "raw": "",
                        "err": True,
                    },
                    None,
                    None,
                    None,
                )

            backoff = (
                0.8
                * (1.6 ** attempt)
                + random.uniform(
                    0,
                    0.25,
                )
            )

            await asyncio.sleep(
                min(
                    backoff,
                    8.0,
                )
            )


async def _worker_ml(
    index: int,
    prompt: str,
    results: list,
    ttft_list: list,
    e2e_list: list,
    request_ids: list,
):
    result, ttft_ms, e2e_ms, request_id = (
        await _call_with_retry_multi(
            prompt
        )
    )

    results[index] = result
    ttft_list[index] = ttft_ms
    e2e_list[index] = e2e_ms
    request_ids[index] = request_id


async def _run_wave_ml(
    prompts_wave: list,
    start_index: int,
    results: list,
    ttft_list: list,
    e2e_list: list,
    request_ids: list,
):
    coroutines = [
        _worker_ml(
            start_index + i,
            prompt,
            results,
            ttft_list,
            e2e_list,
            request_ids,
        )
        for i, prompt
        in enumerate(prompts_wave)
    ]

    step = max(
        1,
        len(coroutines)
        // PROGRESS_DIV,
    )

    completed = 0

    for future in as_completed(
        coroutines
    ):
        await future
        completed += 1

        if (
            completed % step == 0
            or completed
            == len(coroutines)
        ):
            print(
                f"[TEST] Wave progress "
                f"{completed}/{len(coroutines)} "
                f"({completed / len(coroutines):.1%})",
                flush=True,
            )


async def run_concurrent_ml(
    prompts_all: list,
):
    total = len(
        prompts_all
    )

    results = [None] * total
    ttft_ms = [None] * total
    e2e_ms = [None] * total
    request_ids = [None] * total

    if total <= WAVE_SIZE:
        print(
            f"[TEST] Running single wave "
            f"of {total} with "
            f"concurrency={MAX_CONCURRENCY}",
            flush=True,
        )

        await _run_wave_ml(
            prompts_all,
            0,
            results,
            ttft_ms,
            e2e_ms,
            request_ids,
        )

        return (
            results,
            ttft_ms,
            e2e_ms,
            request_ids,
        )

    print(
        f"[TEST] Total {total} anchors. "
        f"Waves of {WAVE_SIZE} with "
        f"concurrency={MAX_CONCURRENCY}",
        flush=True,
    )

    start = 0
    wave_number = 1

    while start < total:
        end = min(
            start + WAVE_SIZE,
            total,
        )

        print(
            f"[TEST] Starting wave "
            f"{wave_number}: "
            f"rows {start}..{end - 1}",
            flush=True,
        )

        await _run_wave_ml(
            prompts_all[start:end],
            start,
            results,
            ttft_ms,
            e2e_ms,
            request_ids,
        )

        print(
            f"[TEST] Finished wave "
            f"{wave_number} "
            f"({end}/{total} overall, "
            f"{end / total:.1%})",
            flush=True,
        )

        start = end
        wave_number += 1

    return (
        results,
        ttft_ms,
        e2e_ms,
        request_ids,
    )


def _run(coroutine):
    """Run an asynchronous task safely inside a notebook."""
    try:
        loop = asyncio.get_running_loop()

    except RuntimeError:
        return asyncio.run(
            coroutine
        )

    else:
        import nest_asyncio

        nest_asyncio.apply()

        return loop.run_until_complete(
            coroutine
        )

### Execute Test Inference

Run the concurrent inference pipeline for all test anchors and collect the model outputs, latency measurements, and request identifiers.

In [ ]:
assert "multi_prompt" in anchors_ml.columns, (
    "Expected 'multi_prompt' in anchors_ml."
)

prompts = anchors_ml[
    "multi_prompt"
].tolist()

print(
    f"[RUN] anchors={len(anchors_ml):,} "
    f"| prompts={len(prompts):,}",
    flush=True,
)

all_results, ttft_ms_list, e2e_ms_list, req_ids = _run(
    run_concurrent_ml(prompts)
)

### Attach Predictions to Test Anchors

Validate the four-event prediction payloads, summarize predicted positive rates, and attach the binary predictions to the corresponding test anchors.

In [ ]:
assert len(all_results) == len(anchors_ml), (
    "The number of inference results does not match the number of test anchors."
)

invalid_payloads = 0

positive_counts = {
    key: 0
    for key in EXPECTED_KEYS
}

prediction_rows = []

for result in all_results:
    valid = (
        isinstance(result, dict)
        and "preds" in result
        and isinstance(result["preds"], dict)
    )

    if not valid:
        invalid_payloads += 1

    predictions = (
        result["preds"]
        if valid
        else {
            key: 0
            for key in EXPECTED_KEYS
        }
    )

    predictions = {
        key: int(
            predictions.get(
                key,
                0,
            )
        )
        for key in EXPECTED_KEYS
    }

    for key in EXPECTED_KEYS:
        positive_counts[key] += (
            predictions[key]
        )

    prediction_rows.append(
        predictions
    )


print(
    f"[SCHEMA] invalid payloads: "
    f"{invalid_payloads}/{len(all_results)}"
)

for key, count in positive_counts.items():
    rate = (
        100.0
        * count
        / max(
            1,
            len(all_results),
        )
    )

    print(
        f"[RATE] {key}: "
        f"ones={count}/{len(all_results)} "
        f"({rate:.1f}%)"
    )


preds_df = pd.DataFrame(
    prediction_rows
)[EXPECTED_KEYS].astype(int)

preds_df.columns = [
    f"{key}_pred_any_7d"
    for key in EXPECTED_KEYS
]

anchors_ml = (
    anchors_ml
    .reset_index(drop=True)
)

anchors_ml = pd.concat(
    [
        anchors_ml,
        preds_df,
    ],
    axis=1,
)

print(
    "[ATTACH] Added 4 prediction columns "
    f"to anchors_ml ({len(anchors_ml):,} rows)."
)

### Summarize and Save Inference Latency

Compute median and 95th-percentile inference timing statistics and save the per-request latency measurements for the test set.

In [ ]:
def _percentile_safe(values, percentile):
    valid_values = [
        value
        for value in values
        if (
            value is not None
            and np.isfinite(value)
        )
    ]

    if not valid_values:
        return None

    return float(
        np.percentile(
            np.array(
                valid_values,
                dtype=float,
            ),
            percentile,
        )
    )


ttft_p50 = _percentile_safe(
    ttft_ms_list,
    50,
)

ttft_p95 = _percentile_safe(
    ttft_ms_list,
    95,
)

e2e_p50 = _percentile_safe(
    e2e_ms_list,
    50,
)

e2e_p95 = _percentile_safe(
    e2e_ms_list,
    95,
)


print(
    "\n=== Latency (milliseconds) ===",
    flush=True,
)

print(
    f"TTFT p50="
    f"{None if ttft_p50 is None else round(ttft_p50, 1)} "
    f"| p95="
    f"{None if ttft_p95 is None else round(ttft_p95, 1)}",
    flush=True,
)

print(
    f"E2E p50="
    f"{None if e2e_p50 is None else round(e2e_p50, 1)} "
    f"| p95="
    f"{None if e2e_p95 is None else round(e2e_p95, 1)}",
    flush=True,
)


assert len(ttft_ms_list) == len(anchors_ml)
assert len(e2e_ms_list) == len(anchors_ml)
assert len(req_ids) == len(anchors_ml)


safe_model_name = (
    MODEL_NAME
    .replace(":", "_")
    .replace("/", "_")
)

lat_path = os.path.join(
    RESULTS_DIR,
    f"latency_TEST_{safe_model_name}_multilabel.csv",
)


lat_df = pd.DataFrame(
    {
        "idx": np.arange(
            len(all_results),
            dtype=int,
        ),
        "region_id": (
            anchors_ml["region_id"]
            .astype(int)
            .tolist()
        ),
        "anchor_date": (
            pd.to_datetime(
                anchors_ml[DATE_COL],
                errors="coerce",
            )
            .dt.strftime("%Y-%m-%d")
            .tolist()
        ),
        "ttft_ms": ttft_ms_list,
        "e2e_ms": e2e_ms_list,
        "request_id": req_ids,
    }
)

lat_df.to_csv(
    lat_path,
    index=False,
)

print(
    f"[LATENCY] Saved per-request latency data: "
    f"{lat_path}",
    flush=True,
)

### Evaluate Multi-Label Predictions

Align the four event predictions with the corresponding ground-truth labels and compute per-event and overall metrics for the ANY-in-7 prediction task.

In [ ]:
# Prediction columns from the test-anchor results
pred_cols = [
    f"{event_type}_pred_any_7d"
    for event_type in EVENT_CATEGORIES
]

preds_wide = anchors_ml[
    ["region_id", DATE_COL] + pred_cols
].copy()


# Convert ground-truth labels from long to wide format
labels_wide = (
    sel_df
    .pivot_table(
        index=["region_id", DATE_COL],
        columns="event_type",
        values="label_7d",
        aggfunc="max",
    )
    .reindex(
        columns=EVENT_CATEGORIES
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

labels_wide.columns.name = None

labels_wide = labels_wide.rename(
    columns={
        event_type: f"{event_type}_label_7d"
        for event_type in EVENT_CATEGORIES
    }
)


# Align predictions and ground-truth labels
eval_df = preds_wide.merge(
    labels_wide,
    on=["region_id", DATE_COL],
    how="inner",
    validate="one_to_one",
)

assert len(eval_df) == len(preds_wide), (
    "Some prediction anchors could not be matched to ground-truth labels."
)


def evaluate_hard_any7d(df):
    rows = []
    all_y = []
    all_pred = []

    for event_type in EVENT_CATEGORIES:
        y_true = (
            df[f"{event_type}_label_7d"]
            .astype(int)
            .values
        )

        y_pred = (
            df[f"{event_type}_pred_any_7d"]
            .astype(int)
            .values
        )

        precision = precision_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        )

        recall = recall_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        )

        f1 = f1_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        )

        rows.append(
            {
                "event_type": event_type,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "pred_pos_rate": y_pred.mean(),
                "true_pos_rate": y_true.mean(),
            }
        )

        all_y.append(
            y_true
        )

        all_pred.append(
            y_pred
        )


    per_event = pd.DataFrame(
        rows,
        columns=[
            "event_type",
            "precision",
            "recall",
            "f1",
            "pred_pos_rate",
            "true_pos_rate",
        ],
    )

    all_y = np.stack(
        all_y,
        axis=1,
    )

    all_pred = np.stack(
        all_pred,
        axis=1,
    )


    # Preserve the metric calculation used in the original evaluation
    micro_f1 = f1_score(
        all_y.ravel(),
        all_pred.ravel(),
        average="micro",
        zero_division=0,
    )

    macro_f1 = (
        per_event["f1"].mean()
    )

    hamming = hamming_loss(
        all_y,
        all_pred,
    )

    subset_accuracy = accuracy_score(
        all_y,
        all_pred,
    )


    overall = pd.DataFrame(
        {
            "micro_f1": [
                micro_f1
            ],
            "macro_f1": [
                macro_f1
            ],
            "hamming_loss": [
                hamming
            ],
            "subset_accuracy": [
                subset_accuracy
            ],
        }
    )


    print(
        "\n=== Evaluation Metrics "
        "(per event type, ANY in 7d) ==="
    )

    print(
        per_event.to_string(
            index=False
        )
    )

    print(
        "\n=== Overall Multi-Label Metrics ==="
    )

    print(
        overall.to_string(
            index=False
        )
    )

    return (
        per_event,
        overall,
    )


per_event_metrics, overall_metrics = (
    evaluate_hard_any7d(
        eval_df
    )
)